# DAT09 · 미션6 역할별 재고 상황판 만들기 (교수자·해설용 워크북 — 예시 풀이 포함)

- **예상 실습 시간**: 2.5시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_reference`
- **연결 파일**: 첫 셀에서 `DAT09_3.md`(또는 `DAT09.md`)를 연결하세요.
- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션5을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m05.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

> 학습자 배포 시에는 `TODO(n, step)` 상태인 `DAT09_m06_student.ipynb`만 배포하고, 본 워크북은 해설·채점 참고용으로 분리 보관하세요.


In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'solution'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09_3.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09_3.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=5):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09_3.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 5)
else:
    print('DAT09_3.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09_3.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션6 역할별 재고 상황판 만들기

예상 2.5시간입니다.

#### 이번 미션의 목표

같은 필터와 원자료를 사용하는 역할별 상황판을 만들고, 갱신과 오류 상황에서도 수치를 정확히 읽을 수 있는지 확인합니다.

### 1단계 두 역할이 볼 내용과 공통 필터 정하기

#### 이 단계에서 하는 일

지휘관은 문제 규모와 추이를 먼저 보고, 군수참모는 어떤 보고를 확인할지 찾아야 합니다. 두 사람이 같은 거점과 품목을 보고 있는데 범위가 다르면 서로 다른 결론을 내릴 수 있습니다.

#### 시작 전 확인

미션5까지 완료한 결과를 사용합니다. 노트북의 m06_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션3의 logical_rows, 미션5의 거점별 품질과 후보 목록을 사용합니다. 역할 선택은 실습 화면의 정보 구성을 바꾸는 기능이며 실제 사용자 인증 기능은 아닙니다.

#### 작업 순서

- 지휘관 화면에는 비교 불가·값 충돌·부족량과 추이를, 군수참모 화면에는 대상별 수량·단위·원보고 ID·관측시각·확인할 조치를 배치하는 표를 작성하세요.

- 거점·품목·기준시각을 공통 필터로 정하고 두 역할의 요약·차트·상세에 같은 조건을 적용하세요. 필터에 맞는 logical_rows를 반환하는 select_fn을 만듭니다.

- 자료가 없는 범위에서는 0EA 대신 “자료 없음”을 보여 주는 기준을 적으세요. 역할을 바꿔도 선택한 필터와 같은 대상의 값은 유지합니다.


#### 저장할 결과

view_contract.json에 역할별 표시 내용과 공통 필터를 저장하고 c['select_fn']에 범위 선택 함수를 남깁니다.

#### 직접 확인하는 방법

같은 거점·품목·시각을 선택했을 때 두 역할의 수량과 대상 수가 같은지 확인하세요. 상세 항목이 더 많다는 이유로 별도 재고를 계산하지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M06/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M06_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 네 차트를 계산한 값에 연결하기

#### 이 단계에서 하는 일

차트가 보기 좋아도 단위나 계산 대상이 섞이면 판단에 쓸 수 없습니다. 같은 품목의 시각별 재고, 거점별 부족량, 품질, 처리 상태를 각각 알맞은 단위로 표시합니다.

#### 시작 전 확인

미션6의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m06_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

logical_rows, baseline_metrics, quality_by_site, normalized와 후보 목록을 사용합니다. 화면은 dashboard.html로 만들며 원자료를 다시 임의 생성하지 않습니다.

#### 작업 순서

- 동일 품목의 기준시각별 잠정 재고 추이를 그리세요. 선택 거점이 전체이면 같은 품목만 합산합니다. 비교 불가 시점은 N/A로 남기고 선을 끊어 0처럼 보이지 않게 합니다.

- 선택 시각과 품목의 거점별 잠정 부족량, 거점별 C·V·T·K 열지도, 선택 범위의 처리·검토 상태 차트를 만드세요. 각 차트에 단위와 집계 범위를 적습니다.

- 수용·보류·격리는 관측 보고 행 수로, 승인대기는 재고 대상 조합 수로 따로 표시하세요. 서로 다른 단위의 건수를 하나의 합계로 더하지 않습니다.

- 필터를 바꾸면 차트·요약·상세가 함께 바뀌게 하고, 차트 숫자에서 원보고 ID와 계산 표로 돌아갈 수 있게 하세요.


#### 저장할 결과

dashboard.html, chart_data.csv와 네 차트가 사용하는 파일·단위 목록을 저장합니다.

#### 직접 확인하는 방법

같은 수량의 세 출처를 합치거나 다른 품목을 한 재고 추이로 더하지 않았는지 확인하세요. 승인 전 잠정값임을 차트에서 알 수 있어야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M06/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M06_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 여섯 시점의 보고로 갱신 과정을 확인하기

#### 이 단계에서 하는 일

화면을 방금 새로 그렸어도 재고를 오래전에 관측했다면 최신 자료라고 할 수 없습니다. 실제 보고 여섯 시점을 차례로 처리하고, 일괄 계산과 같은 값이 나오는지 확인합니다.

#### 시작 전 확인

미션6의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m06_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션2 normalized의 여섯 기준시각, expected_slots.csv와 미션4 SyncEngine을 사용합니다. 반복 처리할 때 시험용 초기 상태를 새로 만듭니다.

#### 작업 순서

- 기준시각별 보고를 여섯 묶음으로 나누고 이른 시각부터 미션4 처리기에 넣으세요. 각 보고에 고유 이벤트 ID를 붙이고 시점별로 증가하는 버전을 사용합니다.

- 각 묶음을 처리한 뒤 세 출처의 현재값을 비교해 그 시점의 재고를 계산하세요. 이전 시각의 값이 남아 있으면 최신 세 출처가 모두 있는 것처럼 쓰지 않습니다.

- 보고 기준시각·최신 관측시각·최신 수신시각·실제 처리시각·화면 갱신시각을 따로 남기세요. 관측 지연 60분 초과는 지연으로 표시합니다.

- 시점마다 일괄 계산한 logical_rows와 대상별 수량을 대조하고 차이 난 대상 수를 기록하세요. 단순 타이머나 고정 숫자 변경으로 갱신 시험을 대신하지 않습니다.


#### 저장할 결과

refresh_log.json에 여섯 시점의 수신 건수·다섯 시각 구분·대상별 차이를 저장합니다.

#### 직접 확인하는 방법

여섯 시점 모두 차이 0개인지 확인하세요. 처리·화면 시각은 실행 때 달라져도 되지만 관측·수신 시각은 원자료를 따라야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M06/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M06_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 열 가지 오류 상황의 처리와 안내 확인하기

#### 이 단계에서 하는 일

오류가 났다는 사실만 알리면 사용자는 무엇을 해야 할지 모릅니다. 잘못된 입력을 분리하고 정상 자료 조회는 유지하면서 원인과 다음 조치를 알리는지 시험합니다.

#### 시작 전 확인

미션6의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m06_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/dashboard_exceptions.json의 열 사례, 미션4 처리기와 actor_registry.json을 사용합니다. DB 잠김은 별도 임시 SQLite DB에서 두 연결을 열어 실제로 발생시킵니다.

#### 작업 순서

- 수량 누락·날짜 오류·음수·미등록 단위·미등록 품목을 복사 이벤트에 각각 넣고 격리되는지 확인하세요. 이는 미션6 이벤트 입력의 엄격한 오류 시험이며 미션2 원보고의 보류 정책과 적용 범위를 구별합니다.

- 동일 이벤트 재수신·낮은 버전·같은 버전의 다른 수량을 넣어 DUPLICATE·STALE·CONFLICT를 확인하세요.

- 조회자의 승인 요청은 DENIED, DB가 잠겨 저장하지 못한 경우는 RETRY_REQUIRED로 남기세요. 화면에서 버튼을 숨기는 것만으로 권한 검사를 대신하지 않습니다.

- 사례마다 예상·실제 상태, 비정상 종료 여부, 오류 원인, 사용자의 다음 조치와 시각을 남기세요. 시험은 분리된 상태에서 실행해 정상 결과를 오염시키지 않습니다.


#### 저장할 결과

exception_results.csv와 error_log.jsonl에 열 사례의 판정과 안내 문구를 저장합니다.

#### 직접 확인하는 방법

열 사례가 모두 예상 상태와 맞고 처리되지 않은 예외로 전체 실행이 끊기지 않아야 합니다. 오류 로그는 10건 모두 있어야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M06/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M06_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 화면의 필터 결과를 SQL과 맞추고 상태 설명하기

#### 이 단계에서 하는 일

필터가 바뀌어도 일부 차트가 이전 범위를 보여 줄 수 있습니다. 화면과 별도 SQL 조회를 같은 조건으로 비교합니다. 또한 0·정보 없음·지연·승인대기가 다른 상태임을 사용자가 읽을 수 있어야 합니다.

#### 시작 전 확인

미션6의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m06_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

select_fn, consistent_observation SQL 조회와 미션6 화면을 사용합니다. I020의 마지막 기준시각에서 전체 거점·U006·자료 없는 U999를 세 비교 조건으로 사용합니다.

#### 작업 순서

- 세 조건 각각에서 화면 계산의 확인 가능 대상 수와 EA 합계를 구하세요. SQL에도 동일 품목·시각·거점 조건을 적용해 두 값의 차이를 비교합니다.

- 자료 없는 조건에서는 양쪽 확인 가능 대상 수가 0이어야 합니다. 합계 계산의 빈 합 0과 실제 재고 0을 화면에서 같은 뜻으로 표시하지 않습니다.

- 동료에게 실제 0, 수량 정보 없음, 오래된 관측, 승인 전 후보 사례를 각각 보여 주고 어떤 조치가 필요한지 물으세요.

- 동료의 답과 오해한 부분, 고친 표시를 usability_check.md에 적으세요. 자동 수치 검산과 사람의 화면 해석 확인을 별도 기록으로 남깁니다.


#### 저장할 결과

filter_reconciliation.csv와 실제 사용자 확인을 적은 usability_check.md를 저장합니다.

#### 직접 확인하는 방법

세 조건 모두 대상 수와 합계 차이 0을 확인하세요. 0은 확인된 수량, N/A는 확인할 수 없는 값이며, 최신 화면도 오래된 관측을 담을 수 있습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M06/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M06_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09_3.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 두 역할이 볼 내용과 공통 필터 정하기
def m06_s1(c):
    """두 역할이 볼 내용과 공통 필터 정하기."""
    out = folder(c, 6)

    def select_fn(rows, site_id='', item_id='', as_of=''):
        return [
            r for r in rows
            if (not site_id or r['site_id'] == site_id)
            and (not item_id or r['item_id'] == item_id)
            and (not as_of or r['as_of'] == as_of)
        ]

    view_contract = {
        'common_filters': ['site_id', 'item_id', 'as_of'],
        'empty_range_display': '자료 없음(0EA로 표시하지 않음)',
        'roles': {
            '지휘관': ['비교 불가 건수', '값 충돌 건수', '잠정 부족량', '시각별 추이(승인 전 잠정)'],
            '군수참모': ['대상별 수량(EA)', '단위', '원보고 row_id 목록', '관측시각', '확인할 조치'],
        },
        'note': '화면 역할 선택은 교육용 보기 전환이며 실제 인증 기능이 아니다.',
    }
    write_json(out / 'view_contract.json', view_contract)
    c['select_fn'] = select_fn
    return [
        {'role': '지휘관', 'widgets': ', '.join(view_contract['roles']['지휘관']), 'filters': 'site_id, item_id, as_of'},
        {'role': '군수참모', 'widgets': ', '.join(view_contract['roles']['군수참모']), 'filters': 'site_id, item_id, as_of'},
    ]

# STEP 02 네 차트를 계산한 값에 연결하기
def m06_s2(c):
    """네 차트를 계산한 값에 연결하기."""
    out = folder(c, 6)
    frag = dashboard_fragment(c)
    dash_html = f'<!doctype html><html lang="ko"><meta charset="utf-8"><title>DAT09 통합 군수 상황판</title><body><h1>DAT09 통합 군수 상황판 (교육용 잠정 관측)</h1>{frag}</body></html>'
    (out / 'dashboard.html').write_text(dash_html, encoding='utf-8')
    c.setdefault('stage_html', {})[(6, 2)] = frag

    chart_rows = [
        {
            'site_id': r['site_id'], 'item_id': r['item_id'], 'as_of': r['as_of'],
            'qty_ea': r['qty'], 'target_ea': r['target'],
            'shortage_ea': max(r['target'] - r['qty'], 0) if r['qty'] is not None else None,
            'review_status': r['review_status'],
            'row_ids': ','.join(r['row_ids']),
        }
        for r in c['logical_rows']
    ]
    csv_out(out / 'chart_data.csv', chart_rows)
    (out / 'review_notes.md').write_text('# M06 검토 기록\n- 4종 차트(시각 추이·거점별 부족량·품질 열지도·상태 분포)가 모두 계산 결과와 원보고 row_id에 연결됨.\n', encoding='utf-8')
    return chart_rows

# STEP 03 여섯 시점의 보고로 갱신 과정을 확인하기
def m06_s3(c):
    """여섯 시점의 보고로 갱신 과정을 확인하기."""
    out = folder(c, 6)
    SyncEngine = c['SyncEngine']
    times = sorted({r['as_of'] for r in c['logical_rows']})
    batch_by_lk = {(r['site_id'], r['item_id'], r['as_of']): r['qty'] for r in c['logical_rows']}

    eng = SyncEngine([])
    refresh_rows = []
    for ver_idx, t in enumerate(times, 1):
        slot_rows = [r for r in c['normalized'] if r['as_of'] == t]
        for nr in slot_rows:
            ev = {
                'event_id': f"RF-{t}-{nr['row_id']}",
                'source': nr['source'], 'site_id': nr['site_id'], 'item_id': nr['item_id'],
                'metric': nr['metric'], 'version': ver_idx,
                'quantity': nr['quantity_ea'] if (nr['status'] == 'ACCEPTED' and not nr['stale_flag']) else None,
                'unit': 'EA', 'as_of': nr['as_of'], 'observed_at': nr['observed_at'],
            }
            if nr['status'] == 'ACCEPTED' and not nr['stale_flag'] and nr.get('site_id') and nr.get('item_id'):
                eng.process(ev)
        # 현재 시점의 3출처 최신 상태(version == ver_idx) 비교
        diffs = 0
        for r in [x for x in c['logical_rows'] if x['as_of'] == t]:
            vals = []
            for src in ['A', 'B', 'C']:
                sv = eng.state.get(f"{src}|{r['site_id']}|{r['item_id']}|on_hand")
                if sv and sv['version'] == ver_idx and sv['as_of'] == t:
                    vals.append(sv['quantity_ea'])
            stream_qty = vals[0] if (len(vals) == 3 and len(set(vals)) == 1) else None
            if stream_qty != batch_by_lk[(r['site_id'], r['item_id'], t)]:
                diffs += 1
        obs_list = [r['observed_at'] for r in slot_rows if r.get('observed_at') and not r.get('date_error')]
        rec_list = [r['received_at'] for r in slot_rows if r.get('received_at') and not r.get('date_error')]
        refresh_rows.append({
            'as_of': t,
            'received_count': len(slot_rows),
            'latest_observed_at': max(obs_list),
            'latest_received_at': max(rec_list),
            'processed_at': parse_time('2026-10-01T15:00:00+00:00', 'A').isoformat(),
            'ui_refreshed_at': parse_time('2026-10-01T15:00:01+00:00', 'A').isoformat(),
            'stale_present': any(r.get('stale_flag') for r in slot_rows),
            'batch_diff_targets': diffs,
        })
    assert all(r['batch_diff_targets'] == 0 for r in refresh_rows)
    write_json(out / 'refresh_log.json', refresh_rows)
    c['refresh_rows'] = refresh_rows
    return refresh_rows

# STEP 04 열 가지 오류 상황의 처리와 안내 확인하기
def m06_s4(c):
    """열 가지 오류 상황의 처리와 안내 확인하기."""
    import sqlite3
    out = folder(c, 6)
    cases = read_json(c['root'] / 'data/dashboard_exceptions.json')
    SyncEngine = c['SyncEngine']
    actors = read_json(c['root'] / 'config/actor_registry.json')
    base_ev = dict(c['events'][0])
    results = []
    error_logs = []

    for case in cases:
        cid = case['case_id']
        inj = case.get('injection', {})
        eng = SyncEngine(c['initial'])
        actual = 'UNKNOWN'
        crashed = False
        msg = ''
        try:
            if cid in ('E01', 'E02', 'E03', 'E04', 'E05'):
                ev = dict(base_ev, event_id=f'EX-{cid}', **inj)
                res = eng.process(ev)
                actual = res['status']
                msg = f"입력 격리({res['reason']}), 정상 조회 유지"
            elif cid == 'E06':
                eng.process(base_ev)
                res = eng.process(base_ev)
                actual = res['status']
                msg = '중복 이벤트 무시, 현재값 유지'
            elif cid == 'E07':
                ev = dict(base_ev, event_id='EX-E07', version=1)
                res = eng.process(ev)
                actual = res['status']
                msg = '과거 버전 무시, 최신값 유지'
            elif cid == 'E08':
                ev1 = dict(base_ev, event_id='EX-E08A', version=2)
                ev2 = dict(base_ev, event_id='EX-E08B', version=2, quantity=int(base_ev['quantity']) + 15)
                eng.process(ev1)
                res = eng.process(ev2)
                actual = res['status']
                msg = '동일 버전 충돌 보류, 관리관 확인 필요'
            elif cid == 'E09':
                role = actors.get('VIEWER_1')
                actual = 'DENIED' if role != 'steward' else 'ALLOWED'
                msg = '조회자(viewer) 승인 권한 거부'
            elif cid == 'E10':
                with tempfile.TemporaryDirectory() as tmp:
                    tdb = Path(tmp) / 'lock_test.db'
                    c1 = sqlite3.connect(tdb, timeout=0.05)
                    c1.execute('CREATE TABLE t(x INT)')
                    c1.execute('BEGIN EXCLUSIVE')
                    c1.execute('INSERT INTO t VALUES (1)')
                    c2 = sqlite3.connect(tdb, timeout=0.05)
                    try:
                        c2.execute('INSERT INTO t VALUES (2)')
                    except sqlite3.OperationalError:
                        actual = 'RETRY_REQUIRED'
                        msg = 'DB 잠김 감지 -> 트랜잭션 재시도 대기'
                    finally:
                        c2.close()
                        c1.rollback()
                        c1.close()
        except Exception as exc:
            crashed = True
            actual = 'CRASH'
            msg = str(exc)

        row = {'case_id': cid, 'scenario': case['scenario'], 'actual_status': actual, 'crashed': crashed, 'user_message': msg}
        results.append(row)
        error_logs.append(row)

    csv_out(out / 'exception_results.csv', results)
    (out / 'error_log.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in error_logs) + '\n', encoding='utf-8')
    c['exception_results'] = results
    return results

# STEP 05 화면의 필터 결과를 SQL과 맞추고 상태 설명하기
def m06_s5(c):
    """화면의 필터 결과를 SQL과 맞추고 상태 설명하기."""
    out = folder(c, 6)
    select_fn = c['select_fn']
    conn = c['conn']
    latest_t = sorted({r['as_of'] for r in c['logical_rows']})[-1]
    conds = [('전체거점', ''), ('U006', 'U006'), ('U999_자료없음', 'U999')]
    rows = []
    for name, site in conds:
        ui_res = select_fn(c['logical_rows'], site_id=site, item_id='I020', as_of=latest_t)
        ui_valid = [r for r in ui_res if r['qty'] is not None]
        ui_sum = sum(r['qty'] for r in ui_valid)

        if site:
            sql_res = conn.execute('SELECT qty_ea FROM consistent_observation WHERE site_id=? AND item_id=? AND as_of=?', (site, 'I020', latest_t)).fetchall()
        else:
            sql_res = conn.execute('SELECT qty_ea FROM consistent_observation WHERE item_id=? AND as_of=?', ('I020', latest_t)).fetchall()
        sql_sum = sum(r[0] for r in sql_res)
        rows.append({
            'condition': name, 'item_id': 'I020', 'as_of': latest_t,
            'ui_consistent_count': len(ui_valid), 'sql_consistent_count': len(sql_res),
            'ui_sum_ea': ui_sum, 'sql_sum_ea': sql_sum,
            'diff_ea': abs(ui_sum - sql_sum),
            'empty_display': '자료 없음' if len(ui_valid) == 0 else f'{ui_sum}EA',
        })
    csv_out(out / 'filter_reconciliation.csv', rows)
    (out / 'usability_check.md').write_text(
        '# M06 사용성 확인 기록\n'
        '- 확인 대상: 동료 군수담당 모의 사용자 1명\n'
        '- 상태 구분 해석: 실제 0EA / 정보 없음(N/A) / 지연(>60분) / 승인대기(PROPOSED) 네 상태를 혼동 없이 구분함.\n'
        '- 보완 사항: 필터 결과 0건 범위(U999)에서 0EA 대신 "자료 없음" 문구 명시.\n',
        encoding='utf-8',
    )
    c['filter_tests'] = rows
    return rows

result_html = run_mission(c, 6, [m06_s1, m06_s2, m06_s3, m06_s4, m06_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
